___

# <font color= #EF4444> **Speech Radicalization Analysis** </font>
#### <font color= #f0565e> `Deep Learning`</font>
<Strong> Sofía Maldonado, Oscar Josué Rocha & Viviana Toledo </Strong>

_12/05/2026._

___

# <font color= #EF4444> **Import Libraries & Data** </font>

In [ ]:
# General
import numpy as np
import pandas as pd
from datasets import load_from_disk

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score

# Modeling
import optuna
from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import make_scorer, f1_score

In [ ]:
dataset = load_from_disk('../data/processed/bertweet_embeddings')
dataset

Dataset({
    features: ['embeddings', 'labels'],
    num_rows: 6218
})

# <font color= #EF4444> **Data Exploration** </font>

In [ ]:
# Extract features and labels from dataset
X = np.array(dataset['embeddings'])
y = np.array(dataset['labels'])

# Print embeddings and labels size
print(f'Embeddings: {X.shape}')
print(f'Target: {y.shape}')

Embeddings: (6218, 768)
Target: (6218,)


The dataset contains 6,218 classified samples, and the embedding dimensions are 768. Let's check the labels in the dataset: 

In [ ]:
np.unique(y)

array(['Dangerous Speech', 'Enraged Speech', 'Extreme Speech',
       'Hate Grammar', 'Hate Speech', 'Horror Grammar'], dtype='<U16')

And now, let's calculate the mean and variation of the embeddings:

In [ ]:
norms = np.linalg.norm(X, axis=1)

print(f'Norm of 100 Samples: {norms[:100]}\n')
print(f'Mean: {norms.mean()}')
print(f'Std: {norms.std()}')

Norm of 10 Samples: [9.20206515 9.02520785 9.29831318 9.18525175 9.14438607 9.24426894
 9.24675102 9.12241138 9.00705607 9.22589057]

Mean: 9.096391631278419
Std: 0.08907492632261334


Variation within embeddings is not great, showcased by the standard deviation (0.08). Additionally, the embeddings are tightly packed, probably because of the common 'hate' thematic. Since we're going to be using an XGBoost model, and tree-based models are not particularly sensitive to scaling, the data will not be scaled.

# <font color= #EF4444> **Preprocessing** </font>

To start, let's map the labels to use as future reference:

In [ ]:
# Initialize label encoder
label_encoder = LabelEncoder()

# Fit label encoder
y_label = label_encoder.fit_transform(y)

# Show label mapping
mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
mapping

{np.str_('Dangerous Speech'): np.int64(0),
 np.str_('Enraged Speech'): np.int64(1),
 np.str_('Extreme Speech'): np.int64(2),
 np.str_('Hate Grammar'): np.int64(3),
 np.str_('Hate Speech'): np.int64(4),
 np.str_('Horror Grammar'): np.int64(5)}

And now, let's do a train-test-split:

In [58]:
X_train, X_test, y_train, y_test = train_test_split(X, y_label, test_size=0.2, random_state=69, stratify=y)

# <font color= #EF4444> **Modeling** </font>

A bayesian optimization method was chosen for hyperparameter selection, through the optuna library. This library requires to define an objective function to optimize, as well as the model's hyperparameters:

In [64]:
def objective(trial):

    params = {
        # Parameters
        "objective": "multi:softprob",
        "num_class": len(np.unique(y_label)),
        "eval_metric": "mlogloss",

        # Hyperparameters
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
        "gamma": trial.suggest_float("gamma", 0, 3),

        # Penalizations for overfitting
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10, log=True),

        # General Parameters
        "tree_method": "hist",
        "random_state": 42,
        "device": 'cuda',
    }

    model = XGBClassifier(**params)

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="f1_macro",         # Maximize f1-macro for unbalanced classes
    )

    return scores.mean()

With the trials defined, let's run 50 of them:

In [65]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=50
)

[I 2026-05-10 21:05:13,632] A new study created in memory with name: no-name-e26457db-209e-4287-b9b1-2920c6e4b9da
[I 2026-05-10 21:07:33,867] Trial 0 finished with value: 0.25280941562509207 and parameters: {'n_estimators': 519, 'max_depth': 6, 'learning_rate': 0.024184994165730257, 'min_child_weight': 1, 'gamma': 1.1160540048113878, 'subsample': 0.5493682369464516, 'reg_alpha': 1.3060838784226397e-07, 'reg_lambda': 1.761834912480747}. Best is trial 0 with value: 0.25280941562509207.
[I 2026-05-10 21:09:15,992] Trial 1 finished with value: 0.24260341596729385 and parameters: {'n_estimators': 363, 'max_depth': 6, 'learning_rate': 0.02275203172296557, 'min_child_weight': 4, 'gamma': 1.3679811971981959, 'subsample': 0.5170193325507866, 'reg_alpha': 1.311048537398116e-08, 'reg_lambda': 2.3458458466676377e-07}. Best is trial 0 with value: 0.25280941562509207.
[I 2026-05-10 21:09:59,042] Trial 2 finished with value: 0.2515597196941364 and parameters: {'n_estimators': 539, 'max_depth': 3, 'le

KeyboardInterrupt: 

In [66]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

clf = LogisticRegression(
    max_iter=3000,
    n_jobs=-1
)

scores = cross_val_score(
    clf,
    X,
    y_label,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

print(scores.mean())

/home/vivienne/apps/deep-learning/speech_radicalization_analysis/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/vivienne/apps/deep-learning/speech_radicalization_analysis/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/vivienne/apps/deep-learning/speech_radicalization_analysis/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/vivienne/apps/deep-learning/speech_radicali

0.2969190399835121
